# Task-Based EEG GAN Analysis for HBN Dataset

This notebook demonstrates the complete workflow for analyzing task-based EEG recordings from the Healthy Brain Network dataset using Wasserstein GANs.

## Workflow
1. Load and explore task-based EEG data
2. Preprocess and extract segments
3. Train task-specific GANs
4. Evaluate synthetic vs real data
5. Compare task characteristics

In [ ]:
import sys
import os

# Add local modules
sys.path.insert(0, '.')

from task_based_eeg_preprocessing import (
    HBNTaskDataLoader,
    TaskEEGPreprocessor,
    TaskEEGDataManager,
    process_all_hbn_tasks
)

from task_based_eeg_gan import (
    TaskGANTrainer,
    train_task_gans,
    EEGSegmentDataset
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## Step 1: Explore Available Data

In [ ]:
# Define data paths (modify for your environment)
RAW_DATA_DIR = './raw/ds005515'  # Location of downloaded .set files
PROCESSED_DIR = './task_gan_data'  # Where to save preprocessed segments
MODEL_DIR = './task_gan_models'  # Where to save trained models

# Check if raw data exists
if os.path.exists(RAW_DATA_DIR):
    print(f"✓ Raw data directory exists: {RAW_DATA_DIR}")
    loader = HBNTaskDataLoader(RAW_DATA_DIR)
    print(f"\nTask file summary:")
    print(loader.get_task_metadata())
else:
    print(f"✗ Raw data directory not found: {RAW_DATA_DIR}")
    print("  Use AWS S3 to download: aws s3 sync s3://openneuro.org/ds005515/ ./raw/ds005515 --no-sign-request")

## Step 2: Preprocess All Tasks

This extracts segments from each task and saves them as normalized arrays ready for GAN training.

In [ ]:
# Run full preprocessing pipeline
# NOTE: This may take 10-30 minutes depending on number of files

preprocessing_results = process_all_hbn_tasks(
    raw_data_dir=RAW_DATA_DIR,
    output_dir=PROCESSED_DIR,
    segment_duration=2.0,  # 2-second segments
    max_files_per_task=None  # Set to 5 for quick test
)

## Step 3: Inspect Preprocessed Data

In [ ]:
data_manager = TaskEEGDataManager(PROCESSED_DIR)
task_summary = data_manager.get_task_summary()
print("Preprocessed data summary:")
print(task_summary)

# Load one task to inspect
if len(task_summary) > 0:
    first_task = task_summary.iloc[0]['task']
    segments, metadata = data_manager.load_task_segments(first_task)
    
    print(f"\nFirst task ({first_task}):")
    print(f"  Segments shape: {segments.shape}")
    print(f"  Channels: {metadata['n_channels']}")
    print(f"  Timepoints: {metadata['n_timepoints']}")
    print(f"  Sampling rate: {metadata['sfreq']} Hz")
    print(f"  Channel names: {metadata['ch_names'][:5]}... (showing first 5)")

## Step 4: Train Task-Based GANs

Trains a separate Wasserstein GAN for each task.

In [ ]:
# Determine device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Train GANs
# NOTE: This will take 30-60 minutes on CPU, 5-10 minutes on GPU
gan_results = train_task_gans(
    preprocessed_data_dir=PROCESSED_DIR,
    output_dir=MODEL_DIR,
    n_epochs=50,  # Increase to 100+ for production
    device=device,
    skip_tasks=[]  # Can skip specific tasks if needed
)

## Step 5: Compare Task-Based GAN Performance

In [ ]:
# Create summary dataframe
summary_data = []
for task, res in gan_results.items():
    summary_data.append({
        'task': task,
        'n_segments': res['n_segments'],
        'g_loss': res['train_result']['final_g_loss'],
        'd_loss': res['train_result']['final_d_loss'],
        'mmd': res['mmd']
    })

results_df = pd.DataFrame(summary_data)
print("\nGAN Training Results:")
print(results_df.to_string())

# Plot results
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Segments per task
axes[0].bar(results_df['task'], results_df['n_segments'], color='steelblue')
axes[0].set_title('Number of Segments per Task')
axes[0].set_ylabel('Segments')
axes[0].tick_params(axis='x', rotation=45)

# Final losses
axes[1].plot(results_df['task'], results_df['g_loss'], 'o-', label='Generator Loss', marker='o')
axes[1].plot(results_df['task'], results_df['d_loss'], 's-', label='Discriminator Loss', marker='s')
axes[1].set_title('Final GAN Losses')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)

# MMD (distribution similarity)
axes[2].bar(results_df['task'], results_df['mmd'], color='coral')
axes[2].set_title('Maximum Mean Discrepancy\n(Lower = Better Match)')
axes[2].set_ylabel('MMD')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('task_gan_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nPlot saved as 'task_gan_comparison.png'")

## Step 6: Generate Synthetic EEG and Compare

Generate synthetic EEG from trained GANs and compare with real data.

In [ ]:
# Select a task to analyze in detail
selected_task = results_df.loc[results_df['mmd'].idxmin(), 'task']  # Best performing task
print(f"Selected task for detailed analysis: {selected_task}")

# Load real data
real_segments, metadata = data_manager.load_task_segments(selected_task)
print(f"Real segments shape: {real_segments.shape}")

# Create trainer and load model
trainer = TaskGANTrainer(
    segments=real_segments,
    task_name=selected_task,
    device=device
)

# Load trained weights
model_path = os.path.join(MODEL_DIR, f'{selected_task}_generator.pt')
trainer.generator.load_state_dict(torch.load(model_path, map_location=device))
print(f"Loaded generator from {model_path}")

# Generate synthetic samples
n_synthetic = 100
synthetic = trainer.generate_samples(n_samples=n_synthetic)
print(f"Generated {len(synthetic)} synthetic samples: {synthetic.shape}")

## Step 7: Visualize Real vs Synthetic EEG

In [ ]:
# Reshape for visualization
n_channels = metadata['n_channels']
n_timepoints = metadata['n_timepoints']

real_reshaped = real_segments[:50].reshape(-1, n_channels, n_timepoints)
synthetic_reshaped = synthetic[:50].reshape(-1, n_channels, n_timepoints)

# Plot sample real vs synthetic
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):
    # Real
    axes[0, i].plot(real_reshaped[i*5].T, alpha=0.7)
    axes[0, i].set_title(f'Real Segment {i+1}')
    axes[0, i].set_ylabel('Amplitude')
    
    # Synthetic
    axes[1, i].plot(synthetic_reshaped[i*5].T, alpha=0.7)
    axes[1, i].set_title(f'Synthetic Segment {i+1}')
    axes[1, i].set_ylabel('Amplitude')

plt.suptitle(f'{selected_task} - Real vs Synthetic EEG Segments', fontsize=14)
plt.tight_layout()
plt.savefig(f'{selected_task}_real_vs_synthetic.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Plot saved as '{selected_task}_real_vs_synthetic.png'")

## Step 8: Spectral Analysis

In [ ]:
from scipy.signal import welch

# Compute power spectral density
sfreq = metadata['sfreq']

# Real data
real_flat = real_reshaped[:20].reshape(-1, n_timepoints)
freqs, psd_real = welch(real_flat, sfreq, nperseg=min(256, n_timepoints))
psd_real_mean = psd_real.mean(axis=0)

# Synthetic data
synthetic_flat = synthetic_reshaped[:20].reshape(-1, n_timepoints)
freqs, psd_synthetic = welch(synthetic_flat, sfreq, nperseg=min(256, n_timepoints))
psd_synthetic_mean = psd_synthetic.mean(axis=0)

# Plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.semilogy(freqs, psd_real_mean, label='Real', linewidth=2)
plt.semilogy(freqs, psd_synthetic_mean, label='Synthetic', linewidth=2, linestyle='--')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power Spectral Density')
plt.title(f'{selected_task} - Power Spectrum')
plt.legend()
plt.grid(True, alpha=0.3)

# Difference
plt.subplot(1, 2, 2)
diff = np.abs(psd_real_mean - psd_synthetic_mean) / (psd_real_mean + 1e-8)
plt.plot(freqs, diff, color='red', linewidth=2)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Relative Difference')
plt.title('Real vs Synthetic Difference')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{selected_task}_spectral_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9: Summary and Statistics

In [ ]:
print(f"\n{'='*80}")
print("Task-Based EEG GAN Analysis Summary")
print(f"{'='*80}")

print(f"\nPreprocessed Data:")
print(f"  Total segments: {task_summary['n_segments'].sum()}")
print(f"  Tasks processed: {len(task_summary)}")
print(f"  EEG channels: {task_summary.iloc[0]['n_channels']}")
print(f"  Segment duration: {metadata['segment_duration']}s @ {metadata['sfreq']}Hz = {metadata['n_timepoints']} timepoints")

print(f"\nGAN Training:")
print(f"  Models trained: {len(gan_results)}")
print(f"  Average G Loss: {results_df['g_loss'].mean():.4f}")
print(f"  Average D Loss: {results_df['d_loss'].mean():.4f}")
print(f"  Average MMD: {results_df['mmd'].mean():.4f}")

print(f"\nBest Performing Tasks:")
top_tasks = results_df.nsmallest(3, 'mmd')
for idx, row in top_tasks.iterrows():
    print(f"  {row['task']:25s} - MMD: {row['mmd']:.4f}")

print(f"\nOutput Directories:")
print(f"  Preprocessed data: {PROCESSED_DIR}")
print(f"  Trained models: {MODEL_DIR}")
print(f"\n{'='*80}")

## Transfer to Longleaf HPC

To transfer this analysis to Longleaf:

```bash
# 1. Copy Python modules and notebook
scp task_based_eeg_preprocessing.py <onyen>@longleaf.unc.edu:~/project/
scp task_based_eeg_gan.py <onyen>@longleaf.unc.edu:~/project/
scp task_based_eeg_analysis.ipynb <onyen>@longleaf.unc.edu:~/project/

# 2. Download full HBN dataset to Longleaf
# SSH into Longleaf, then:
module load aws-cli/2.13.30
aws s3 sync s3://openneuro.org/ds005515/sub-*/eeg ./raw/ds005515 --no-sign-request

# 3. Run pipeline with HPC
# See longleaf_setup.md for SLURM submission scripts
```